# 02 — $Q_{net}$ decomposition from GEOS atmosphere diagnostics

Reconstruct the net downward surface heat flux from the GEOS surface collections,

$$Q_{net} = SW_{net} + LW_{net} - LH - SH,$$

with GEOS/MERRA-2 conventions (radiative terms `SWGNT`, `LWGNT` positive **down**;
turbulent terms `EFLUX`, `HFLUX` positive **up**), and check closure against the
ocean-side `oceQnet` on a common 1° grid.

**Caveats.** (i) The atmosphere (c1440 cubed-sphere) and ocean (LLC2160) grids differ, so
the comparison uses conservative-in-the-mean bin-averaging of both to 1°. (ii) Under sea
ice, `oceQnet` includes ice–ocean exchange and is not expected to match the
atmosphere-side surface flux — high latitudes are excluded from the closure statistics.
(iii) GEOS fluxes over land are irrelevant here and masked by the ocean comparison.

In [ ]:
# Environment check: this notebook must run on SciServer (Kraken domain,
# Oceanography image, "Poseidon DYAMOND (ceph)" data volume), or with
# DYAMOND_ROOT pointing at a local subset.
from dyamond_fluxes import dyamond_root

root = dyamond_root()  # raises with setup instructions if the data volume is absent
print(f"DYAMOND root: {root}")

In [ ]:
from dyamond_fluxes import find_stores_with

GEOS_VARS = ["EFLUX", "HFLUX", "SWGNT", "LWGNT"]
geos_hits = find_stores_with(GEOS_VARS)
geos_hits = {s: v for s, v in geos_hits.items() if set(v) & set(GEOS_VARS)}

if not geos_hits:
    raise RuntimeError(
        "No GEOS atmosphere flux collections (EFLUX/HFLUX/SWGNT/LWGNT) found under "
        f"{root}. Fall back to the non-solar residual in notebook 01, or obtain the "
        "GEOS collections from the NCCS Dataportal: "
        "https://gmao.gsfc.nasa.gov/global_mesoscale/dyamond_phaseII/data_access/"
    )
for s, v in geos_hits.items():
    print(s, "->", v)

In [ ]:
import xarray as xr

from dyamond_fluxes import open_store

# The four variables may live in one store or be split across collections
# (e.g., turbulent fluxes vs. radiation); merge whatever notebook 00 found.
geos = xr.merge(
    [open_store(s)[v] for s, v in geos_hits.items()], compat="override", join="inner"
)
geos

In [ ]:
SNAPSHOT = "2020-07-15T12:00"
geos_snap = geos.sel(time=SNAPSHOT, method="nearest")
print("GEOS snapshot:", geos_snap.time.values)

In [ ]:
from dyamond_fluxes import qnet_from_components

decomp = qnet_from_components(
    swgnt=geos_snap["SWGNT"],
    lwgnt=geos_snap["LWGNT"],
    eflux=geos_snap["EFLUX"],
    hflux=geos_snap["HFLUX"],
)
decomp

In [ ]:
# Locate 2-D lat/lon coordinates on the cubed-sphere store (names vary by
# collection: lons/lats, longitude/latitude, ...). Adjust if inspection of the
# dataset above shows different names.
def find_coord(ds, candidates):
    for name in candidates:
        if name in ds.coords or name in ds:
            return ds[name]
    raise KeyError(f"none of {candidates} found; inspect the dataset and set manually")

atm_lon = find_coord(geos_snap, ["lons", "lon", "longitude", "XC"])
atm_lat = find_coord(geos_snap, ["lats", "lat", "latitude", "YC"])
atm_lon, atm_lat = xr.broadcast(atm_lon, atm_lat)
print(atm_lon.dims, atm_lon.shape)

In [ ]:
from dyamond_fluxes import bin_to_latlon, nonsolar_flux, to_positive_down

DLON = DLAT = 1.0

# Atmosphere-side components, binned to 1 deg.
binned = {
    name: bin_to_latlon(decomp[name].load(), atm_lon, atm_lat, dlon=DLON, dlat=DLAT)
    for name in ["shortwave", "longwave", "latent", "sensible", "qnet"]
}

# Ocean-side Qnet on the same grid.
ocean_store = next(iter(find_stores_with(["oceQnet"])))
ds_ocn = open_store(ocean_store)
snap_ocn = ds_ocn.sel(time=SNAPSHOT, method="nearest")
qnet_ocn = to_positive_down(snap_ocn["oceQnet"])
if "Depth" in ds_ocn:
    qnet_ocn = qnet_ocn.where(ds_ocn["Depth"] > 0)
qnet_ocn_binned = bin_to_latlon(
    qnet_ocn.load(), ds_ocn["XC"], ds_ocn["YC"], area=ds_ocn.get("rA"), dlon=DLON, dlat=DLAT
)
print("ocean snapshot:", snap_ocn.time.values)

In [ ]:
from pathlib import Path

import cmocean
import matplotlib.pyplot as plt

FIGDIR = Path("../figures")
FIGDIR.mkdir(exist_ok=True)

fig, axes = plt.subplots(2, 2, figsize=(14, 7), sharex=True, sharey=True)
titles = ["shortwave", "longwave", "latent", "sensible"]
for ax, name in zip(axes.ravel(), titles):
    da = binned[name].where(qnet_ocn_binned.notnull())  # ocean points only
    pc = ax.pcolormesh(da.lon, da.lat, da, cmap=cmocean.cm.balance, vmin=-300, vmax=300)
    ax.set_title(f"{name} (positive down)")
fig.colorbar(pc, ax=axes, shrink=0.8, label="W m$^{-2}$")
fig.suptitle(f"GEOS surface heat flux components, {SNAPSHOT}")
fig.savefig(FIGDIR / "qnet_components_1deg.png", dpi=200, bbox_inches="tight")

In [ ]:
import numpy as np

# Closure: atmosphere-side sum vs ocean-side oceQnet, open ocean equatorward of 60 deg
# (sea ice contaminates the comparison poleward of that).
diff = (binned["qnet"] - qnet_ocn_binned).where(abs(qnet_ocn_binned.lat) < 60)

fig, ax = plt.subplots(figsize=(10, 4.5))
pc = ax.pcolormesh(diff.lon, diff.lat, diff, cmap=cmocean.cm.balance, vmin=-100, vmax=100)
fig.colorbar(pc, ax=ax, label="W m$^{-2}$")
ax.set_title("GEOS (SW+LW-LH-SH) minus ocean oceQnet, 1° bins")
fig.savefig(FIGDIR / "qnet_closure_map.png", dpi=200, bbox_inches="tight")

vals = diff.values[np.isfinite(diff.values)]
print(f"closure residual: mean {vals.mean():.2f}, RMS {np.sqrt((vals**2).mean()):.2f} W m-2")

A small mean residual is expected even for a perfect decomposition: the two sides are
sampled on different grids and possibly at slightly different diagnostic times/averaging
windows, and `oceQnet` includes coupler-side adjustments (e.g., under-ice fluxes,
snow/runoff enthalpy). Large systematic patterns, by contrast, indicate a sign-convention
or variable-selection error — recheck against notebook 00.